# TARA RAG Pipeline v2.0
### ISO/SAE 21434-compliant Threat Analysis & Risk Assessment Generator
**Optimized:** Smart chunking | reports_db ingestion | fixed paths | dedup guard

## 1. DEPENDENCIES INSTALLATION

In [ ]:
%%bash
apt-get install -y libpango-1.0-0 libcairo2 -q
pip install -q haystack-ai
pip install -q "sentence-transformers>=2.2.0"
pip install -q google-ai-haystack
pip install -q lxml
echo 'All dependencies installed.'

## 2. LIBRARY IMPORTS

In [ ]:
import os
import json
import glob
import re
import uuid
import xml.etree.ElementTree as ET
from lxml import etree
from pathlib import Path
from getpass import getpass
from collections import Counter

from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.generators.google_ai import GoogleAIGeminiGenerator

print("✅ Imports complete.")

## 3. PATH CONFIGURATION

In [ ]:
# =============================================
# DATASET PATHS — adjust BASE_PATH if needed
# =============================================
BASE_PATH     = Path("datasets")

MITRE_MOBILE  = BASE_PATH / "mobileattack.json"
MITRE_ICS     = BASE_PATH / "icsattack.json"
ATM_PATH      = BASE_PATH / "atm.json"
CAPEC_PATH    = BASE_PATH / "capec.xml"
CWE_PATH      = BASE_PATH / "cwec.xml"
ECU_PATH      = BASE_PATH / "dataecu.json"
ANNEX_PATH    = BASE_PATH / "annex.json"
CLAUSE_PATH   = BASE_PATH / "clauses"
REPORTS_PATH  = BASE_PATH / "reports_db"

# =============================================
# EMBEDDING MODEL (cost-effective + retrieval-optimised)
# BGE-small beats MiniLM on BEIR benchmarks, same size
# =============================================
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
# Fallback: "sentence-transformers/all-MiniLM-L6-v2"

MAX_CHARS     = 1500   # max chars per chunk for threat-framework entries

# Verify paths exist
for p in [MITRE_MOBILE, MITRE_ICS, ATM_PATH, CAPEC_PATH, CWE_PATH, ANNEX_PATH, CLAUSE_PATH, REPORTS_PATH]:
    status = "✅" if p.exists() else "❌ MISSING"
    print(f"{status}  {p}")

## 4. COMPONENT SETUP — Document Store & Embedders

In [ ]:
document_store = InMemoryDocumentStore()
print("🧠 InMemoryDocumentStore initialized.")

doc_embedder  = SentenceTransformersDocumentEmbedder(model=EMBED_MODEL)
text_embedder = SentenceTransformersTextEmbedder(model=EMBED_MODEL)

doc_embedder.warm_up()
text_embedder.warm_up()
print(f"✅ Embedders ready  [{EMBED_MODEL}]")

# top_k=20: enough context, less noise vs the old top_k=40
retriever = InMemoryEmbeddingRetriever(document_store=document_store, top_k=20)
print("🔍 Retriever created (top_k=20).")

## 5. INGESTION FUNCTIONS
### 5a. Threat Framework Sources (MITRE / ATM / CAPEC / CWE)

In [ ]:
def _truncate(text: str, max_chars: int = MAX_CHARS) -> str:
    """Trim long descriptions to avoid embedding token waste."""
    if len(text) <= max_chars:
        return text
    # Cut at last sentence boundary within limit
    cut = text[:max_chars]
    last_period = cut.rfind(". ")
    return cut[:last_period + 1] if last_period > 0 else cut + "..."


def ingest_mitre(path, source):
    """Load MITRE ATT&CK JSON — one Document per attack-pattern."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    docs = []
    for obj in data.get("objects", []):
        if obj.get("type") != "attack-pattern":
            continue
        name = obj.get("name", "")
        desc = _truncate(obj.get("description", ""))
        if not desc:
            continue
        docs.append(Document(
            content=f"MITRE Technique: {name}\nDescription: {desc}",
            meta={"source": source, "stix_id": obj.get("id")}
        ))
    return docs


def ingest_atm(path):
    """Load Automotive Threat Matrix JSON — one Document per attack-pattern."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    docs = []
    for obj in data.get("objects", []):
        if obj.get("type") != "attack-pattern":
            continue
        name = obj.get("name", "")
        # Strip HTML tags from ATM descriptions
        raw_desc = obj.get("description", "")
        desc = _truncate(re.sub(r"<[^>]+>", " ", raw_desc).strip())
        if not desc:
            continue
        docs.append(Document(
            content=f"ATM Technique: {name}\nDescription: {desc}",
            meta={"source": "ATM", "stix_id": obj.get("id")}
        ))
    return docs


def ingest_capec(xml_path):
    """Load CAPEC XML — one Document per Attack_Pattern."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ns = {"capec": "http://capec.mitre.org/capec-3"}
    docs = []
    for ap in root.findall(".//capec:Attack_Pattern", ns):
        cid  = ap.get("ID")
        name = ap.findtext("capec:Name", default="", namespaces=ns)
        desc = _truncate(ap.findtext("capec:Description", default="", namespaces=ns))
        if not desc:
            continue
        docs.append(Document(
            content=f"CAPEC-{cid}: {name}\nDescription: {desc}",
            meta={"source": "CAPEC", "capec_id": cid}
        ))
    return docs


def ingest_cwe(xml_path):
    """Load CWE XML — one Document per Weakness."""
    parser = etree.XMLParser(recover=True, huge_tree=True)
    tree   = etree.parse(xml_path, parser)
    root   = tree.getroot()
    ns     = {"cwe": "http://cwe.mitre.org/cwe-7"}
    docs   = []
    for w in root.findall(".//cwe:Weakness", namespaces=ns):
        cwe_id = w.get("ID")
        name   = w.get("Name", "")
        desc_el = w.find("cwe:Description", namespaces=ns)
        desc    = _truncate((desc_el.text or "").strip()) if desc_el is not None else ""
        if not desc:
            continue
        docs.append(Document(
            content=f"CWE-{cwe_id}: {name}\nDescription: {desc}",
            meta={"source": "CWE", "cwe_id": f"CWE-{cwe_id}"}
        ))
    return docs


print("✅ Threat-framework ingestion functions defined.")

### 5b. ISO 21434 Clauses & Annex F — Section-Level Chunking

In [ ]:
def _flatten_list(lst):
    """Recursively flatten nested lists to a single list of strings."""
    result = []
    for item in lst:
        if isinstance(item, list):
            result.extend(_flatten_list(item))
        elif isinstance(item, str):
            result.append(item)
    return result


def _section_to_text(section, clause_id=""):
    """Convert a clause section dict to a readable text block."""
    parts = []
    sid   = section.get("section_id", "")
    title = section.get("section_title", "")
    parts.append(f"ISO 21434 Clause {clause_id} — Section {sid}: {title}")

    # Content bullets
    for item in _flatten_list(section.get("content", [])):
        parts.append(f"  - {item}")

    # Requirements
    for req in section.get("requirements", []):
        rid   = req.get("id", "")
        rdesc = " ".join(_flatten_list(req.get("description", [])))
        parts.append(f"  [{rid}] {rdesc}")

    # Recommendations
    for rec in section.get("recommendations", []):
        rid   = rec.get("id", "")
        rdesc = " ".join(_flatten_list(rec.get("description", [])))
        parts.append(f"  [{rid}] (Recommendation) {rdesc}")

    # Nested subsections
    for sub in section.get("subsections", []):
        parts.append(_section_to_text(sub, clause_id))

    return "\n".join(parts)


def ingest_iso_clauses(clause_dir):
    """
    Load ISO 21434 clause JSON files.
    CHUNKING: One Document per section (not per file) for fine-grained retrieval.
    """
    docs  = []
    files = sorted(Path(clause_dir).glob("clause-*.json"))
    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            clause = json.load(f)

        clause_id  = clause.get("clause_id", file.stem.split("-")[1])
        clause_title = clause.get("clause_title", f"Clause {clause_id}")

        for section in clause.get("sections", []):
            text = _section_to_text(section, clause_id)
            if len(text.strip()) < 20:
                continue
            docs.append(Document(
                content=text,
                meta={
                    "source":       "ISO_21434",
                    "clause":       clause_id,
                    "clause_title": clause_title,
                    "section_id":   section.get("section_id", ""),
                    "title":        section.get("section_title", "")
                }
            ))
    return docs


def ingest_annex(annex_path):
    """
    Load Annex F JSON.
    CHUNKING: One Document per annex section.
    """
    if not Path(annex_path).exists():
        print("⚠️  Annex file not found — skipping.")
        return []

    with open(annex_path, "r", encoding="utf-8") as f:
        annex = json.load(f)

    annex_id    = annex.get("annex_id", "F")
    annex_title = annex.get("annex_title", "Guidelines for Impact Rating")
    docs = []

    for section in annex.get("sections", []):
        parts = []
        sid   = section.get("section_id", "")
        stitle = section.get("section_title", "")
        parts.append(f"Annex {annex_id}: {annex_title} — Section {sid}: {stitle}")

        for c in section.get("content", []):
            parts.append(f"  - {c}")

        for t in section.get("tables", []):
            cols = t.get("columns", [])
            rows = t.get("rows", [])
            parts.append(f"\nTable {t.get('table_id')}: {t.get('table_title')}")
            if cols and rows:
                parts.append(" | ".join(cols))
                parts.append("-" * 40)
                for r in rows:
                    parts.append(" | ".join(str(r.get(col, "")) for col in cols))

        for note in section.get("notes", []):
            parts.append(f"Note: {note}")

        text = "\n".join(parts)
        if len(text.strip()) < 20:
            continue
        docs.append(Document(
            content=text,
            meta={
                "source":     "ANNEX_F",
                "annex":      annex_id,
                "section_id": sid,
                "title":      stitle
            }
        ))
    return docs


print("✅ ISO clause & Annex ingestion functions defined.")

### 5c. ECU Data & Reports DB — Fine-Grained Asset/Scenario Chunking

In [ ]:
def ingest_ecu(ecu_path):
    """Load ECU/system data JSON — one Document per ECU entry."""
    if not Path(ecu_path).exists():
        print(f"  ECU file not found: {ecu_path} — skipping.")
        return []
    with open(ecu_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    docs = []
    items = data.items() if isinstance(data, dict) else enumerate(data)
    for key, val in items:
        docs.append(Document(
            content=json.dumps(val, indent=2),
            meta={"source": "ECU", "title": str(key)}
        ))
    return docs


def _clean_node_for_text(node):
    """Extract readable label+description from a node dict."""
    data  = node.get("data", {})
    label = data.get("label", node.get("id", ""))
    desc  = data.get("description", "")
    props = node.get("properties", [])
    ntype = node.get("type", "component")
    return label, desc, props, ntype


def ingest_reports_db(reports_path):
    """
    Ingest TARA reference reports from reports_db/.

    CHUNKING STRATEGY:
      * Each named node (all components, not just isAsset=True) -> 1 Document
      * Each damage derivation entry                            -> 1 Document
      * Each damage detail entry                                -> 1 Document

    All named nodes are ingested so the retriever surfaces
    sub-components and interfaces as reference patterns.
    """
    docs = []
    path = Path(reports_path)
    if not path.exists():
        print(f"  Reports DB folder not found: {reports_path}")
        return docs

    for json_file in sorted(path.glob("*.json")):
        fname = json_file.name
        with open(json_file, "r", encoding="utf-8") as f:
            report = json.load(f)

        # Format A: direct {assets, damage_scenarios}  (infotainment_1.json)
        if "assets" in report and "damage_scenarios" in report:
            assets_block    = report["assets"]
            damage_block    = report["damage_scenarios"]
            model_name      = assets_block.get("model_id", fname.replace(".json", ""))
            nodes_list      = assets_block.get("template", {}).get("nodes", [])
            derivation_list = damage_block.get("Derivations") or damage_block.get("derivation") or []
            details_list    = damage_block.get("Details")      or damage_block.get("details")     or []

        # Format B: wrapped (bms_1.json) -- top-level key is "Damage_scenarios"
        elif "Assets" in report:
            a_block    = report["Assets"][0] if report.get("Assets") else {}
            model_name = report["Models"][0]["name"] if report.get("Models") else fname
            nodes_list = a_block.get("template", {}).get("nodes", [])
            ds_list    = (report.get("Damage_scenarios")
                          or report.get("DamageScenarios")
                          or [])
            d_block         = ds_list[0] if ds_list else {}
            derivation_list = d_block.get("Derivations") or d_block.get("derivation") or []
            details_list    = d_block.get("Details")      or d_block.get("details")     or []
        else:
            print(f"  Unrecognised format in {fname} — skipping.")
            continue

        # Chunk 1: All named nodes (isAsset filter removed)
        node_count = 0
        for node in nodes_list:
            label, desc, props, ntype = _clean_node_for_text(node)
            if not label or label.strip() == "":
                continue
            is_asset = node.get("isAsset", False)
            content = (
                f"Reference Component [{model_name}]: {label}\n"
                f"Type: {ntype}  IsAsset: {is_asset}\n"
                f"Description: {desc}\n"
                f"Security Properties: {\', \'.join(props) if props else \'N/A\'}"
            )
            docs.append(Document(
                content=content,
                meta={"source": "REPORTS_DB", "file": fname,
                      "model": model_name, "type": "asset",
                      "is_asset": is_asset, "node_id": node.get("id", "")}
            ))
            node_count += 1

        # Chunk 2: Damage derivation entries
        deriv_count = 0
        for d in derivation_list:
            name  = d.get("name", "")
            asset = d.get("asset", "")
            loss  = d.get("loss", "")
            scene = d.get("damage_scene", "")
            if not name:
                continue
            content = (
                f"Reference Damage Derivation [{model_name}]:\n"
                f"Threat/Weakness: {name}\n"
                f"Affected Asset: {asset}\n"
                f"Cyber Loss: {loss}\n"
                f"Damage Scene: {scene}"
            )
            docs.append(Document(
                content=content,
                meta={"source": "REPORTS_DB", "file": fname,
                      "model": model_name, "type": "damage_derivation"}
            ))
            deriv_count += 1

        # Chunk 3: Damage detail entries
        detail_count = 0
        for det in details_list:
            dname   = det.get("Name", "")
            ddesc   = _truncate(det.get("Description", ""), 800)
            impacts = det.get("impacts", {})
            losses  = [(cl.get("name", ""), cl.get("node", ""))
                       for cl in det.get("cyberLosses", [])]
            if not dname:
                continue
            impact_str = "  ".join(f"{k}: {v}" for k, v in impacts.items() if v)
            loss_str   = ", ".join(f"{n} ({nd})" for n, nd in losses if n)
            content = (
                f"Reference Damage Scenario [{model_name}]: {dname}\n"
                f"Description: {ddesc}\n"
                f"Cyber Losses: {loss_str}\n"
                f"Impact Ratings: {impact_str}"
            )
            docs.append(Document(
                content=content,
                meta={"source": "REPORTS_DB", "file": fname,
                      "model": model_name, "type": "damage_detail"}
            ))
            detail_count += 1

        print(f"  {fname}: {node_count} components | {deriv_count} derivations | {detail_count} details")

    print(f"\nREPORTS_DB total chunks: {len(docs)}")
    return docs


print("ECU & Reports DB ingestion functions defined.")

## 6. LOAD ALL SOURCES & RUN INGESTION

In [ ]:
# ─── 6a. Threat Frameworks ─────────────────────────────────────────
print("Loading threat frameworks...")
mitre_docs = []
mitre_docs += ingest_mitre(MITRE_MOBILE, "MITRE_MOBILE")
mitre_docs += ingest_mitre(MITRE_ICS,    "MITRE_ICS")
atm_docs   = ingest_atm(ATM_PATH)
capec_docs = ingest_capec(CAPEC_PATH)
cwe_docs   = ingest_cwe(CWE_PATH)
print(Counter(d.meta["source"] for d in mitre_docs + atm_docs + capec_docs + cwe_docs))

# ─── 6b. ISO 21434 Clauses & Annex ─────────────────────────────────
print("\nLoading ISO 21434 clauses...")
iso_docs   = ingest_iso_clauses(CLAUSE_PATH)
annex_docs = ingest_annex(ANNEX_PATH)
print(f"  ISO 21434 sections : {len(iso_docs)}")
print(f"  Annex F sections   : {len(annex_docs)}")

# ─── 6c. ECU Data ───────────────────────────────────────────────────
print("\nLoading ECU data...")
ecu_docs = ingest_ecu(ECU_PATH)
print(f"  ECU entries: {len(ecu_docs)}")

# ─── 6d. REPORTS DB (NEW) ───────────────────────────────────────────
print("\nLoading REPORTS DB (reference TARA reports)...")
reports_docs = ingest_reports_db(REPORTS_PATH)

# ─── 6e. Merge All ──────────────────────────────────────────────────
all_docs  = ecu_docs + iso_docs + annex_docs
all_docs += mitre_docs + atm_docs + capec_docs + cwe_docs
all_docs += reports_docs

print(f"\n{'='*50}")
print(f"Total documents before embedding : {len(all_docs)}")
dist = Counter(d.meta.get("source", "?") for d in all_docs)
for src, cnt in sorted(dist.items()):
    print(f"  {src:<20}: {cnt}")

## 7. EMBED & STORE (with deduplication guard)

In [ ]:
# Guard: skip if already embedded (prevents duplicate docs on cell re-run)
existing = document_store.count_documents()
if existing > 0:
    print(f"ℹ️  Document store already has {existing} documents — skipping re-embedding.")
    print("   To re-embed from scratch, re-run Cell 4 (Component Setup) first.")
else:
    print(f"🔄 Embedding {len(all_docs)} documents...")
    result = doc_embedder.run(documents=all_docs)
    embedded_docs = result["documents"]
    document_store.write_documents(embedded_docs)
    print(f"✅ {document_store.count_documents()} documents embedded and stored.")
    print("\nDistribution:")
    dist = Counter(d.meta.get("source", "?") for d in embedded_docs)
    for src, cnt in sorted(dist.items()):
        print(f"  {src:<20}: {cnt}")

## 8. VERIFICATION ASSERTIONS

In [ ]:
stored = document_store.filter_documents()
sources = {d.meta.get("source") for d in stored}

assert document_store.count_documents() >= 1500, \
    f"Expected ≥1500 docs, got {document_store.count_documents()}"
assert "REPORTS_DB" in sources, \
    "REPORTS_DB source missing — check ingest_reports_db() and REPORTS_PATH"
assert "ISO_21434" in sources, \
    "ISO_21434 source missing — check CLAUSE_PATH and ingest_iso_clauses()"
assert "CWE" in sources and "CAPEC" in sources, \
    "Threat framework sources missing"

reports_count = sum(1 for d in stored if d.meta.get("source") == "REPORTS_DB")
iso_count     = sum(1 for d in stored if d.meta.get("source") == "ISO_21434")

print("✅ All assertions passed!")
print(f"   Total docs   : {document_store.count_documents()}")
print(f"   ISO_21434    : {iso_count} section-level chunks")
print(f"   REPORTS_DB   : {reports_count} asset/scenario chunks")
print(f"   All sources  : {sorted(sources)}")

## 9. API KEY SETUP

In [ ]:
# ⚠ Do NOT hardcode secrets. Use one of these secure methods:

# Method 1 (recommended for Colab): set via Colab secrets panel, then:
# from google.colab import userdata
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Method 2 (environment variable, already set externally):
if "GOOGLE_API_KEY" not in os.environ:
    # Method 3: interactive prompt (never stored in notebook)
    os.environ["GOOGLE_API_KEY"] = getpass("🔐 Enter your Google API key: ")

print("✅ GOOGLE_API_KEY is set.")

## 10. LLM SETUP — Gemini

In [ ]:
if "GOOGLE_API_KEY" not in os.environ:
    raise EnvironmentError("❌ GOOGLE_API_KEY not set — run Cell 9 first.")

generator = GoogleAIGeminiGenerator(
    model="gemini-2.5-flash",   # upgrade from flash-lite for better reasoning
    generation_kwargs={
        "temperature": 0.2,      # low temp for deterministic TARA JSON
        "max_output_tokens": 8192
    }
)

print("🤖 Gemini generator ready.")

## 11. PROMPT TEMPLATE

In [ ]:
template = """
You are an automotive cybersecurity analyst performing Threat Analysis and Risk Assessment (TARA)
according to ISO/SAE 21434 Clause 15.

Your task is to generate a system architecture model and cybersecurity damage scenarios
for the requested automotive ECU or system.

STRICT KNOWLEDGE RULES

- Use ONLY information relevant to the TARGET SYSTEM specified in the SYSTEM REQUEST below.
- Do NOT invent assets or components that are not part of the targeted system.
- If an AUTHORITATIVE ASSET LIST is provided in the request, generate EXACTLY those assets — no additions, no omissions.
  All damage scenarios, derivations, and edges must reference ONLY those listed assets.
- REPORTS_DB entries are structural EXAMPLES ONLY. Do NOT copy their node labels, scenario names, or IDs.
  They show the expected JSON shape, security property patterns, and derivation format.
  All generated content must be derived from the TARGET SYSTEM, not from any reference system (e.g. BMS).
- Do NOT reproduce BMS, Infotainment, or any other system's component names unless the TARGET SYSTEM matches exactly.
- Use realistic automotive architecture relevant to the TARGET SYSTEM only.
- Prefer knowledge retrieved from cybersecurity context (ISO 21434, CWE, CAPEC, MITRE, ATM).
- If information is missing, infer only common industry-standard components for the specified system.

Threat reasoning must follow:
CWE (root weakness) → CAPEC (attack pattern) → MITRE ATT&CK technique → ATM relevance → Damage Scenario

-------------------------------------------------

SYSTEM REQUEST:
{{question}}

CYBERSECURITY KNOWLEDGE CONTEXT:
{% for doc in documents %}
{% if doc.meta.source == "REPORTS_DB" %}
[REFERENCE-PATTERN-ONLY | structural example — do NOT copy node names or scenario content]
{% else %}
[{{ doc.meta.source }}{% if doc.meta.section_id is defined %} § {{ doc.meta.section_id }}{% endif %}{% if doc.meta.type is defined %} | {{ doc.meta.type }}{% endif %}]
{% endif %}
{{ doc.content }}
---
{% endfor %}

-------------------------------------------------

TASK

1. Identify the architecture of the requested system (use the AUTHORITATIVE ASSET LIST if provided).
2. Generate assets that belong strictly to the TARGET SYSTEM — no others.
3. Create architecture relationships (edges) between those assets.
4. Generate realistic cybersecurity damage scenarios referencing only the generated assets.
5. For each damage scenario derive an Impact Rating using SFOP categories.

-------------------------------------------------

IMPACT RATING SCALE

For every damage scenario derive cyber losses using SFOP categories:
Safety | Financial | Operational | Privacy

For each cyber loss assign: Negligible | Minor | Moderate | Major | Severe
Then derive an overall impact rating based on the highest impact.

-------------------------------------------------

STRICT OUTPUT FORMAT

Return ONLY valid JSON. Do not include explanations, markdown fences, or prose.
Start the response with '{'.

Return JSON exactly in this structure:

{
 "assets":{
   "_id":"",
   "user_id":"",
   "model_id":"",

   "template":{
      "nodes":[
         {
           "id":"",
           "type":"default",
           "parentId":"",
           "isAsset":true,
           "data":{
             "label":"",
             "description":"",
             "style":{"backgroundColor":"#dadada","borderColor":"gray","borderStyle":"solid","borderWidth":"2px","color":"black","fontFamily":"Inter","fontSize":"12px","fontWeight":500,"height":50,"width":150}
           },
           "properties":["Integrity","Confidentiality","Availability"],
           "style":{"width":150,"height":50},
           "position":{"x":0,"y":0},
           "positionAbsolute":{"x":0,"y":0},
           "width":150,
           "height":50
         }
      ],

      "edges":[
         {
           "id":"",
           "source":"<source node id>",
           "target":"<target node id>",
           "sourceHandle":"b",
           "targetHandle":"left",
           "type":"step",
           "animated":true,
           "markerEnd":{"color":"#64B5F6","height":18,"type":"arrowclosed","width":18},
           "markerStart":{"color":"#64B5F6","height":18,"orient":"auto-start-reverse","type":"arrowclosed","width":18},
           "style":{"end":true,"start":true,"stroke":"#808080","strokeDasharray":"0","strokeWidth":2},
           "properties":["Integrity"],
           "data":{"label":"","offset":0,"t":0.5}
         }
      ]
   }
 },

 "damage_scenarios":{
   "_id":"",
   "model_id":"",
   "type":"damage",

   "Derivations":[
      {
        "id":"","nodeId":"","task":"Threat Analysis",
        "name":"","loss":"","asset":"",
        "damage_scene":"","isChecked":false
      }
   ],

   "Details":[
      {
        "Name":"",
        "Description":"",
        "cyberLosses":[{"id":"","name":"","node":"","nodeId":"","isSelected":true,"is_risk_added":false}],
        "impacts":{"Financial Impact":"","Safety Impact":"","Operational Impact":"","Privacy Impact":""},
        "key":1,
        "_id":""
      }
   ]
 }
}

-------------------------------------------------

CONSTRAINTS

- Generate ONLY the assets listed in the AUTHORITATIVE ASSET LIST (if provided), or assets strictly belonging to the TARGET SYSTEM.
- Do NOT add components from other ECU systems.
- Damage scenarios must reference valid nodeId values from the nodes above.
- Impact rating must be derived from the damage scenario context.
- Use threat reasoning from CWE, MITRE, CAPEC, ATM — not from REPORTS_DB examples.

Return JSON only. Start the response with '{'.
"""

prompt_builder = PromptBuilder(
    template=template,
    required_variables=["documents", "question"]
)

print("✅ PromptBuilder initialized.")

## 12. BUILD RAG PIPELINE

In [ ]:
tara_pipeline = Pipeline()
tara_pipeline.add_component("text_embedder",  text_embedder)
tara_pipeline.add_component("retriever",      retriever)
tara_pipeline.add_component("prompt_builder", prompt_builder)
tara_pipeline.add_component("llm",            generator)

tara_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
tara_pipeline.connect("retriever",               "prompt_builder.documents")
tara_pipeline.connect("prompt_builder",          "llm")

print("✅ TARA RAG pipeline built and connected successfully.")

## 13. RUN TARA GENERATION

In [ ]:
# ── Set your query here ──────────────────────────────────────────────────────
user_query = "Battery Management System ECU"   # Change as needed
# Examples:
#   "Infotainment Head Unit"
#   "Gateway / Domain Controller"
#   "ADAS ECU"
#   "Telematics Control Unit"
#   "OBD-II Diagnostic Port"
#   "BCM (Body Control Module)"
#   "ABS (Anti-lock Braking System)"
# ─────────────────────────────────────────────────────────────────────────────
print(f"Query set: {user_query}")
print("Run the next cell (13b) to resolve ECU and generate the TARA report.")

### 13b. ECU Resolution from dataecu.json

In [ ]:
# ── Resolve the queried system against dataecu.json ──────────────────────────
# dataecu.json has 50 ECU/system entries, each with a curated `hint` field
# listing the EXACT expected assets for that system. We match user_query to
# the closest entry and inject its hint as an authoritative constraint into the
# LLM prompt -- so the LLM generates ONLY those assets and no others.
# ─────────────────────────────────────────────────────────────────────────────

# Generic automotive suffix words removed before acronym extraction
_SUFFIX_WORDS = {
    "ecu", "system", "module", "interface", "controller", "unit",
    "network", "port", "server", "bus", "head", "vehicle", "automotive"
}

# Well-known phrase -> dataecu.json key aliases  (handles ambiguous queries)
_ALIASES = {
    "obd": "obd",
    "obd-ii": "obd",
    "obd2":  "obd",
    "tcu":   "tcu",
    "telematics control": "tcu",
    "bcm":   "bcm",
    "ecm":   "ecm",
    "ivi":   "ivi",
    "eps":   "eps",
    "abs":   "abs",
    "bms":   "bms",
    "adas":  "adas",
}

def _acronym(text):
    """Generate acronym stripping generic automotive suffixes."""
    skip = {"the", "and", "for", "of", "a", "an", "or", "in", "on", "to", "/"}
    words = [w.strip("()/-").lower() for w in text.replace("/", " ").split()]
    sig_words  = [w for w in words if w and w not in skip]
    core_words = [w for w in sig_words if w not in _SUFFIX_WORDS]
    chosen = core_words if core_words else sig_words
    return "".join(w[0] for w in chosen if w)

def resolve_ecu(query, ecu_path=ECU_PATH):
    """
    Fuzzy-match query against dataecu.json keys and names.

    Matching order:
      0. Alias table (well-known abbreviations / phrases)
      1. Exact key or key as standalone word in query
      2. Full entry name is substring of query
      3. Acronym of query (suffix-stripped) matches or starts a key
      4. Significant name-word overlap (>= 2 shared core words)
      5. Any key word > 3 chars appears in query

    Returns matched entry dict {name, type, hint} or None.
    """
    with open(ecu_path, "r", encoding="utf-8") as f:
        ecu_db = json.load(f)

    q = query.lower().strip()

    # Pass 0: alias table
    for phrase, key in _ALIASES.items():
        if phrase in q and key in ecu_db:
            return ecu_db[key]

    # Pass 1: exact key or key as standalone word
    for key, entry in ecu_db.items():
        if key == q or f" {key} " in f" {q} ":
            return entry

    # Pass 2: full entry name substring match
    for key, entry in ecu_db.items():
        if entry["name"].lower() in q:
            return entry

    # Pass 3a: exact acronym match
    query_acronym = _acronym(q)
    for key, entry in ecu_db.items():
        if query_acronym and query_acronym == key:
            return entry

    # Pass 3b: acronym starts key (within 1 extra char)
    if len(query_acronym) >= 2:
        for key, entry in ecu_db.items():
            if key.startswith(query_acronym) and len(key) - len(query_acronym) <= 1:
                return entry

    # Pass 4: significant name-word overlap
    for key, entry in ecu_db.items():
        name_words = [w.strip("()/-").lower()
                      for w in entry["name"].replace("/", " ").split()]
        core_name  = [w for w in name_words
                      if len(w) > 2 and w not in _SUFFIX_WORDS]
        if sum(1 for w in core_name if w in q) >= 2:
            return entry

    # Pass 5: any key word > 3 chars in query
    for key, entry in ecu_db.items():
        key_words = key.replace("_", " ").split()
        if any(w in q for w in key_words if len(w) > 3):
            return entry

    return None


ecu_entry = resolve_ecu(user_query)
if ecu_entry:
    print(f"Matched ECU  : {ecu_entry['name']}")
    print(f"Type         : {ecu_entry['type']}")
    print(f"Asset hint   : {ecu_entry['hint']}")
else:
    print("No dataecu.json match found -- will use open-ended generation.")

In [ ]:
# Guard: ensure ecu_entry is defined even if Cell 13b was skipped
if "ecu_entry" not in dir():
    ecu_entry = None

# Build enriched query for the LLM  (embedding still uses plain user_query)
if ecu_entry:
    enriched_query = (
        f"{ecu_entry['name']}\n\n"
        f"AUTHORITATIVE ASSET LIST (from system dataecu specification) — "
        f"generate ONLY these assets, no others:\n"
        f"{ecu_entry['hint']}\n\n"
        f"All threat analysis, damage scenarios, and edges must reference "
        f"ONLY the assets listed above. Do NOT add any other components."
    )
else:
    enriched_query = user_query

print(f"Generating TARA for  : {user_query}")
print(f"ECU match            : {ecu_entry['name'] if ecu_entry else 'None (open-ended)'}")
print(f"Enriched query (120c): {enriched_query[:120]}...")
print(f"Retriever top_k      : 20")
print(f"Embedding model      : {EMBED_MODEL}")

result = tara_pipeline.run(
    {
        "text_embedder": {"text": user_query},          # plain query for retrieval
        "prompt_builder": {"question": enriched_query}  # enriched for LLM
    },
    include_outputs_from=["retriever"]
)

print("Pipeline run complete.")

## 14. RETRIEVED CONTEXT INSPECTION

In [ ]:
ret_docs = result["retriever"]["documents"]
print(f"Documents retrieved: {len(ret_docs)}")
print(f"Sources: {Counter(d.meta.get('source') for d in ret_docs)}\n")

for i, doc in enumerate(ret_docs):
    src  = doc.meta.get("source", "?")
    tag  = doc.meta.get("section_id") or doc.meta.get("type") or doc.meta.get("cwe_id") or ""
    print(f"[{i+1:2d}] {src:<18} {tag:<20} {doc.content[:120].strip()}...")

## 15. JSON OUTPUT

In [ ]:
raw_output = result["llm"]["replies"][0]

# Strip any accidental markdown fences
cleaned = re.sub(r"^```[a-z]*\n?", "", raw_output.strip(), flags=re.MULTILINE)
cleaned = re.sub(r"```$", "", cleaned.strip())

try:
    tara_json = json.loads(cleaned)
    print("✅ Valid JSON output parsed.")
except json.JSONDecodeError as e:
    print(f"⚠️  JSON parse error: {e}")
    print("Raw output (first 500 chars):")
    print(cleaned[:500])
    tara_json = None

# ── UUID STAMPING ────────────────────────────────────────────────────────────
# Replace empty / placeholder / hallucinated IDs with proper uuid4 values.
# ─────────────────────────────────────────────────────────────────────────────
def stamp_uuids(obj):
    """Walk the JSON tree and replace any id/_id/model_id that is empty,
    None, or a placeholder string with a fresh uuid4."""
    ID_KEYS = {"id", "_id", "model_id"}

    def _bad(val):
        if not val:
            return True
        if isinstance(val, str) and (
            "PLACEHOLDER" in val or val.strip() == "" or val.startswith("<")
        ):
            return True
        return False

    def _walk(o):
        if isinstance(o, dict):
            for k, v in list(o.items()):
                if k in ID_KEYS and _bad(v):
                    o[k] = str(uuid.uuid4())
                else:
                    _walk(v)
        elif isinstance(o, list):
            for item in o:
                _walk(item)

    _walk(obj)
    return obj


def crosslink_node_ids(obj):
    """After node UUIDs are stamped, propagate real nodeId values into
    Derivations[].nodeId and Details[].cyberLosses[].nodeId by label-matching."""
    nodes = obj.get("assets", {}).get("template", {}).get("nodes", [])
    label_to_id = {
        n.get("data", {}).get("label", "").lower(): n.get("id")
        for n in nodes if n.get("id")
    }

    for d in obj.get("damage_scenarios", {}).get("Derivations", []):
        nid = d.get("nodeId", "")
        if not nid or str(nid).startswith("<") or "PLACEHOLDER" in str(nid):
            matched = label_to_id.get(d.get("asset", "").lower())
            d["nodeId"] = matched if matched else str(uuid.uuid4())

    for det in obj.get("damage_scenarios", {}).get("Details", []):
        for cl in det.get("cyberLosses", []):
            nid = cl.get("nodeId", "")
            if not nid or str(nid).startswith("<") or "PLACEHOLDER" in str(nid):
                matched = label_to_id.get(cl.get("node", "").lower())
                cl["nodeId"] = matched if matched else str(uuid.uuid4())
            if not cl.get("id") or str(cl.get("id", "")).startswith("<"):
                cl["id"] = str(uuid.uuid4())
    return obj


if tara_json:
    tara_json = stamp_uuids(tara_json)
    tara_json = crosslink_node_ids(tara_json)
    node_count  = len(tara_json.get("assets", {}).get("template", {}).get("nodes", []))
    edge_count  = len(tara_json.get("assets", {}).get("template", {}).get("edges", []))
    deriv_count = len(tara_json.get("damage_scenarios", {}).get("Derivations", []))
    ds_count    = len(tara_json.get("damage_scenarios", {}).get("Details", []))
    print(f"   Nodes         : {node_count}")
    print(f"   Edges         : {edge_count}")
    print(f"   Derivations   : {deriv_count}")
    print(f"   Damage details: {ds_count}")
    print("   IDs           : all IDs stamped as proper uuid4 values")

In [ ]:
# Pretty-print the full JSON output
if tara_json:
    print(json.dumps(tara_json, indent=2))

## 16. SAVE OUTPUT TO FILE

In [ ]:
if tara_json:
    safe_name = re.sub(r"[^a-zA-Z0-9_-]", "_", user_query.strip())
    output_file = f"tara_output_{safe_name}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(tara_json, f, indent=2, ensure_ascii=False)
    print(f"✅ TARA report saved to: {output_file}")
else:
    print("⚠️  No valid JSON to save — check Cell 15 for errors.")